<a href="https://colab.research.google.com/github/Valementajat/PageRank_colab/blob/SparkTesting/pageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
''' pip install kaggle '''

' pip install kaggle '

In [2]:
# install PySpark
!apt-get update # Update apt-get repository.
!apt-get install openjdk-11-jdk-headless -qq > /dev/null # Install Java.
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz # Download Apache Sparks.
!tar xf spark-3.5.1-bin-hadoop3.tgz # Unzip the tgz file.
!pip install -q findspark # Install findspark. Adds PySpark to the System path during runtime.


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,669 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/multiver

In [3]:
import os
os.environ['KAGGLE_USERNAME'] = "XXXXX"
os.environ['KAGGLE_KEY'] = "XXXXX"

# Set environment variables for spark
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

dataset_zip = "new-york-times-articles-comments-2020.zip"
if not os.path.exists(dataset_zip):
  !kaggle datasets download -d  benjaminawd/new-york-times-articles-comments-2020




Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
100% 1.94G/1.95G [00:20<00:00, 173MB/s]
100% 1.95G/1.95G [00:20<00:00, 102MB/s]


In [41]:
# Initialize findspark
import findspark
findspark.init()

# Create a PySpark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext
spark

In [5]:
# Unzip the file independenty
articles = "nyt-articles-2020.csv"
comments = "nyt-comments-2020.csv"

if not os.path.exists(articles) and not os.path.exists(comments):
  !unzip new-york-times-articles-comments-2020.zip

Archive:  new-york-times-articles-comments-2020.zip
  inflating: nyt-articles-2020.csv   
  inflating: nyt-comments-2020.csv   
  inflating: nyt-comments-part0.csv  
  inflating: nyt-comments-part1.csv  
  inflating: nyt-comments-part2.csv  
  inflating: nyt-comments-part3.csv  
  inflating: nyt-comments-part4.csv  
  inflating: nyt-comments-part5.csv  
  inflating: nyt-comments-part6.csv  
  inflating: nyt-comments-part7.csv  
  inflating: nyt-comments-part8.csv  
  inflating: nyt-comments-part9.csv  
  inflating: test.csv                
  inflating: train.csv               


In [45]:
import numpy as np
import pandas as pd





In [7]:
from pyspark.sql import functions as F
articles_df = spark.read.option("header", True).csv("/content/nyt-articles-2020.csv")


#parsing the comments in Spark crash so, we do a bit of trickery
chunk_size=10000
comments_df = None

# I know you're not supposed to do this, but the parsing of the comments .csv didn't want to comply

for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size,  usecols=["userID", "articleID"]):
  # clean current chunk
  chunk = chunk.dropna(subset=["userID", "articleID"]).drop_duplicates()
  # Create spark dataframe
  spark_chunk = spark.createDataFrame(chunk.to_dict("records"))

  # append into one Spark DataFrame
  if comments_df is None:
      comments_df = spark_chunk
  else:
      comments_df = comments_df.unionByName(spark_chunk)

# final dedup in Spark
comments_df = comments_df.dropDuplicates().persist()



In [8]:
# Lets initialize a sample freq at the start so it's easy to spot
# Sample_fraq is the sample of articles we're concidering, you can set this between 0.01 and 1.0
sample_frac = 0.2

sampled_articles = (
    articles_df
    .select(F.col("uniqueID").alias("articleID"))
    .distinct()
    .sample(False, sample_frac, seed=42)
    )


In [9]:


# First map
# I know that with spark I could try the RDD, but I already made the mapReduce implementation
# and I don't have that much time
interactions = comments_df.join(sampled_articles, on="articleID", how="inner")



In [11]:
#Then reduce

# Desided to flip this otherway around as I'm rewriting this again, so now user [articles]
# As this is possibly faster
articleUsers = (
    interactions
    .groupBy("userID")
    .agg(F.collect_set("articleID").alias("articles"))
)

In [13]:
# Lets create the article pairs where the common users are >= 2
article_pairs = (
    interactions.alias("a")
    .join(
        interactions.alias("b"),
        on="userID",
        how="inner"
    )
    .where(F.col("a.articleID") < F.col("b.articleID"))   # avoid self-pairs and reversed duplicates
    .groupBy(
        F.col("a.articleID").alias("article1"),
        F.col("b.articleID").alias("article2")
    )
    .agg(
        F.count("*").alias("shared_user_count"),

    )
    .where(F.col("shared_user_count") >= 2)
)


In [14]:
#Making the connections both ways as we only define a "link" between the pages
adjacency = (
    article_pairs.select(
        F.col("article1").alias("src"),
        F.col("article2").alias("dst")
    )
    .unionByName(
        article_pairs.select(
            F.col("article2").alias("src"),
            F.col("article1").alias("dst")
        )
    )
    .dropDuplicates()
    .cache()
)


In [15]:
# At this point in the night I realize that I didn't have to redo everything with Spark, but distribute the computation of the matrixes.....

In [39]:
pages = (
    interactions
    .select(F.col("articleID").alias("id"))
    .distinct()
    .cache()
)

In [44]:
# degree is practically how many times that page is seen in the connections

outdeg = (
    adjacency
    .groupBy("src")
    .count()
    .withColumnRenamed("count", "outdeg")
    .cache()
)

#sparse transition matrix
# This is already a distributed dataframe so...
# sc.parallelize(connection_matrix).cache() like this won't be neede
connection_matrix = (
    adjacency
    .join(outdeg, on="src", how="left")
    .withColumn("weight", F.lit(1.0) / F.col("outdeg"))
    .select("src", "dst", "weight")
    .cache() # parallelized by this
)

In [ ]:




# variables to make this a bit simpler to modify
N = pages.count()
damping=0.8 # damping var. can be 0.8 to 0.9
max_iterations = 100


tolerance=1.e-5


# The pageRank function itself
for i in range(max_iterations):
    contributions = (
        adjacency
        .join(ranks, adjacency.src == ranks.id, "inner")
        .join(outdeg, "src", "inner")
        .select(
            F.col("dst").alias("id"),
            (F.col("rank") / F.col("outdeg")).alias("contrib")
        )
    )

    ranks = (
        pages
        .join(
            contributions.groupBy("id").agg(F.sum("contrib").alias("sum_contrib")),
            on="id",
            how="left"
        )
        .fillna(0.0, subset=["sum_contrib"])
        .select(
            "id",
            (((1 - damping) / N) + damping * F.col("sum_contrib")).alias("rank")
        )
    )


In [ ]:
''' # Dangling nodes
dangling_nodes = (
    pages
    .join(outdeg, pages.id == outdeg.src, how="left")
    .where(F.col("outdeg").isNull())
    .select("page")
    .cache()
) '''

In [38]:
''' ranks.orderBy(F.col("rank").desc()).show(truncate=False) '''

' ranks.orderBy(F.col("rank").desc()).show(truncate=False) '

In [21]:
''' chunk_size = 10000

# Using a set to avoid duplicates
ids = set()


# To guard against the memory we load the csv in chuncks and process it right away
for article_chunk in pd.read_csv("nyt-articles-2020.csv", chunksize=chunk_size):
   for article_id in article_chunk["uniqueID"]:
      ids.add(article_id)

n = len(ids) '''

' chunk_size = 10000\n\n# Using a set to avoid duplicates\nids = set()\n\n\n# To guard against the memory we load the csv in chuncks and process it right away\nfor article_chunk in pd.read_csv("nyt-articles-2020.csv", chunksize=chunk_size):\n   for article_id in article_chunk["uniqueID"]:\n      ids.add(article_id)\n\nn = len(ids) '

In [22]:
''' # Lets create the dataset we want to work from the articles
sample_size = int(n * sample_frac)

# Adding a seed to make the sampling reproducible
random.seed(10)
sample_ids = random.sample(list(ids), sample_size)
n = len(sample_ids)
# Maybe this will be a bit more efficient to work with than a full on URL?
id_to_i = {article_id: i for i, article_id in enumerate(sample_ids)} '''



' # Lets create the dataset we want to work from the articles\nsample_size = int(n * sample_frac)\n\n# Adding a seed to make the sampling reproducible\nrandom.seed(10)\nsample_ids = random.sample(list(ids), sample_size)\nn = len(sample_ids)\n# Maybe this will be a bit more efficient to work with than a full on URL?\nid_to_i = {article_id: i for i, article_id in enumerate(sample_ids)} '

In [23]:
''' # Creating an empty adjancency matrix
adjencencyMatrix =  np.zeros((n, n ), dtype=float) '''



' # Creating an empty adjancency matrix\nadjencencyMatrix =  np.zeros((n, n ), dtype=float) '

In [24]:
''' # TO avoid blowing up the memory usage, we load the
# comments in chucks
chunk_size = 10000

# First part of the mapReduce, we create the key value pairs
KeyValuePairs = []

for chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size):
  # Start the MapReduce by creating key-value pairs
  for index, row in chunk.iterrows():
    if row['articleID'] in id_to_i:
      # Currently creating this like "article -> user", at now it's okay i guess
      KeyValuePairs.append((id_to_i[row["articleID"]], row["userID"]))

del chunk
del chunk_size

gc.collect()
 '''


' # TO avoid blowing up the memory usage, we load the\n# comments in chucks\nchunk_size = 10000\n\n# First part of the mapReduce, we create the key value pairs\nKeyValuePairs = []\n\nfor chunk in pd.read_csv("nyt-comments-2020.csv", chunksize=chunk_size):\n  # Start the MapReduce by creating key-value pairs\n  for index, row in chunk.iterrows():\n    if row[\'articleID\'] in id_to_i:\n      # Currently creating this like "article -> user", at now it\'s okay i guess\n      KeyValuePairs.append((id_to_i[row["articleID"]], row["userID"]))\n\ndel chunk\ndel chunk_size\n\ngc.collect()\n '

In [25]:
''' # Lets group the KeyValuePairs by the key
KeyValueGroup = defaultdict(set)
for article_idx, user_id in KeyValuePairs:
    # but
    KeyValueGroup[article_idx].add(user_id)
 '''

' # Lets group the KeyValuePairs by the key\nKeyValueGroup = defaultdict(set)\nfor article_idx, user_id in KeyValuePairs:\n    # but\n    KeyValueGroup[article_idx].add(user_id)\n '

In [26]:
''' # Then the final reduce so we can use this
connectionPairs = []

items = list(KeyValueGroup.items())

# This is a possibly a problem?
# I've created the groups with article [uids], but it would be more efficient
# with uid [articles] ?

# I'm checking items against other values in the same set and looking if there
# are two unique intersecting user pairs so we can say that there
for a_idx, users_a in items:
    for b_idx, users_b in items:
        if a_idx != b_idx:
            if len(users_a.intersection(users_b)) >= 2:
                connectionPairs.append((a_idx, b_idx)) '''


" # Then the final reduce so we can use this\nconnectionPairs = []\n\nitems = list(KeyValueGroup.items())\n\n# This is a possibly a problem?\n# I've created the groups with article [uids], but it would be more efficient\n# with uid [articles] ?\n\n# I'm checking items against other values in the same set and looking if there\n# are two unique intersecting user pairs so we can say that there\nfor a_idx, users_a in items:\n    for b_idx, users_b in items:\n        if a_idx != b_idx:\n            if len(users_a.intersection(users_b)) >= 2:\n                connectionPairs.append((a_idx, b_idx)) "

In [27]:
''' for i in range(10):
  print(connectionPairs[i]) '''

' for i in range(10):\n  print(connectionPairs[i]) '

In [28]:
''' # degree is practically how many times that page is seen in the connections
degree = Counter()
for i, j in connectionPairs:
    degree[i] += 1


for i, j in connectionPairs:
    adjencencyMatrix[i, j] = 1.0 / degree[i] '''


' # degree is practically how many times that page is seen in the connections\ndegree = Counter()\nfor i, j in connectionPairs:\n    degree[i] += 1\n\n\nfor i, j in connectionPairs:\n    adjencencyMatrix[i, j] = 1.0 / degree[i] '

In [29]:
''' #We have a problem, dangling nodes, so this is to uniformally
# set the propability to go from a page to every page when calculating the rank

row_sums = adjencencyMatrix.sum(axis=1)
dangling = (row_sums == 0)


# replace each dangling row with uniform distribution
adjencencyMatrix[dangling, :] = 1.0 / n '''

' #We have a problem, dangling nodes, so this is to uniformally\n# set the propability to go from a page to every page when calculating the rank\n\nrow_sums = adjencencyMatrix.sum(axis=1)\ndangling = (row_sums == 0)\n\n\n# replace each dangling row with uniform distribution\nadjencencyMatrix[dangling, :] = 1.0 / n '

In [30]:
''' # So we have our stochastic adjencencyMatrix or connection matrix

# Without any dangling pages (or nodes)

# Lets run the power iteration

# This is the formula to "teleport", during the power iteration.
# as in, do we take the next page from the page's links or do we take one from
# all of the pages
# P = βM + (1 - β)1/N

beta = 0.85 # example size is between 0.8 - 0.9
n = len(adjencencyMatrix)

#now this is the pageRank transition matrix or the google matrix
P = beta * adjencencyMatrix + (1 - beta) / n '''

' # So we have our stochastic adjencencyMatrix or connection matrix\n\n# Without any dangling pages (or nodes)\n\n# Lets run the power iteration\n\n# This is the formula to "teleport", during the power iteration.\n# as in, do we take the next page from the page\'s links or do we take one from\n# all of the pages\n# P = βM + (1 - β)1/N\n\nbeta = 0.85 # example size is between 0.8 - 0.9\nn = len(adjencencyMatrix)\n\n#now this is the pageRank transition matrix or the google matrix\nP = beta * adjencencyMatrix + (1 - beta) / n '

In [31]:
''' # lets initialize the rank vector r^(0) which has every rank as 1/n
r = np.ones(n) / n '''

' # lets initialize the rank vector r^(0) which has every rank as 1/n\nr = np.ones(n) / n '

In [32]:
''' # Now we calculat the new ranks using the transition matrix and the old rank
# r^new =  r^old · A
for i in range(100):
  # python's matrix manipulation to multiply matrix with another matrix
    r = r @ P '''

" # Now we calculat the new ranks using the transition matrix and the old rank\n# r^new =  r^old · A\nfor i in range(100):\n  # python's matrix manipulation to multiply matrix with another matrix\n    r = r @ P "

In [33]:
''' # This is just a sanity check
print(r.sum()) '''

' # This is just a sanity check\nprint(r.sum()) '

In [34]:
''' # Top 10 indices by rank (largest first)
top_k = 10
top_idx = np.argsort(r)[-top_k:][::-1]

print("Top ranks:")
for idx in top_idx:
    print(idx, r[idx]) '''

' # Top 10 indices by rank (largest first)\ntop_k = 10\ntop_idx = np.argsort(r)[-top_k:][::-1]\n\nprint("Top ranks:")\nfor idx in top_idx:\n    print(idx, r[idx]) '

In [35]:
%whos

Variable           Type            Data/Info
--------------------------------------------
F                  module          <module 'pyspark.sql.func<...>yspark/sql/functions.py'>
N                  int             3326
SparkSession       type            <class 'pyspark.sql.session.SparkSession'>
adjacency          DataFrame       DataFrame[src: string, dst: string]
articleUsers       DataFrame       DataFrame[userID: bigint,<...> articles: array<string>]
article_pairs      DataFrame       DataFrame[article1: strin<...>hared_user_count: bigint]
articles           str             nyt-articles-2020.csv
articles_df        DataFrame       DataFrame[newsdesk: strin<...>string, uniqueID: string]
chunk              DataFrame                   userID       <...>\n[4649 rows x 2 columns]
chunk_size         int             10000
comments           str             nyt-comments-2020.csv
comments_df        DataFrame       DataFrame[articleID: string, userID: bigint]
contributions      DataFrame   